# W2/D2 — RCA (Root Cause Analysis)
**Goal:** Build RCA pipeline using Graph Traversal and Retrieval (kNN) to find root cause and suggest remediation.

---

In [ ]:
#Import & Load Data
import json
from pathlib import Path
import networkx as nx

from rca import (
    load_json,
    load_alerts,
    build_graph,
    graph_temporal_scorer,
    retrieve_similar_incidents,
    rca_pipeline
)

# Paths
DATASET_DIR = Path("dataset")
D1_RESULTS_PATH = Path("../../w2/d1/results/cluster_summary.json")
if not D1_RESULTS_PATH.exists():
    D1_RESULTS_PATH = Path("d:/DevopsAndCloud/AIOPS/w2/d1/results/cluster_summary.json")

# Load data
clusters_data = load_json(D1_RESULTS_PATH)
alerts = load_alerts(str(DATASET_DIR / "alerts_sample.jsonl"))
graph = build_graph(str(DATASET_DIR / "services.json"))
incidents = load_json(str(DATASET_DIR / "incidents_history.json"))["incidents"]

print(f"Loaded {len(clusters_data['clusters'])} clusters from D1")
print(f"Loaded {len(incidents)} historical incidents")


Loaded 1 clusters from D1
Loaded 29 historical incidents


In [ ]:
#  Graph + Temporal Scorer
# Process the first cluster
cluster = clusters_data['clusters'][0]
print(f"Processing Cluster: {cluster['cluster_id']}")
print(f"Services involved: {cluster['services']}")
print("-" * 40)

top_candidates = graph_temporal_scorer(cluster, alerts, graph)
print("Graph + Temporal Ranking (Top 3):")
for svc, score in top_candidates[:3]:
    print(f"  {svc:20s}: {score:.2f}")


Processing Cluster: c-000-000
Services involved: ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'recommender-svc', 'search-svc']
----------------------------------------
Graph + Temporal Ranking (Top 3):
  payment-svc         : 0.99
  checkout-svc        : 0.93
  cart-svc            : 0.79


In [ ]:
# Retrieval (kNN-style
similar_incs = retrieve_similar_incidents(cluster, incidents, top_k=3)
print("Top 3 Similar Historical Incidents:")
for score, inc in similar_incs:
    print(f"[{score:.2f}] {inc['id']} | Root Cause: {inc['root_cause_service']} | Class: {inc['root_cause_class']}")
    print(f"       Summary: {inc['summary'][:80]}...")


Top 3 Similar Historical Incidents:
[0.80] INC-2025-11-08 | Root Cause: payment-svc | Class: connection_pool_exhaustion
       Summary: Payment-svc v3.2 deploy at 09:42 leak DB pool. Pool 50/50 used trong 5 phút. Dow...
[0.80] INC-2026-03-20 | Root Cause: edge-lb | Class: ddos
       Summary: Volumetric DDoS 5x normal traffic. Edge-lb saturate, all upstream visible degrad...
[0.60] INC-2025-08-02 | Root Cause: recommender-svc | Class: memory_leak
       Summary: Recommender OOM mỗi 4h sau deploy v3.1. Pandas DataFrame không release giữa requ...


In [ ]:
# Full RCA Pipeline & Output
results = []
for c in clusters_data['clusters']:
    res = rca_pipeline(c, alerts, graph, incidents)
    results.append(res)
    
output = {
    "clusters_analyzed": len(clusters_data['clusters']),
    "results": results
}

out_dir = Path("results")
out_dir.mkdir(exist_ok=True)
out_path = out_dir / "rca_output.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Written RCA results to {out_path}")
print(json.dumps(output['results'][0], indent=2))


Written RCA results to results\rca_output.json
{
  "cluster_id": "c-000-000",
  "graph_top3": [
    [
      "payment-svc",
      0.99
    ],
    [
      "checkout-svc",
      0.93
    ],
    [
      "cart-svc",
      0.79
    ]
  ],
  "root_cause": "payment-svc",
  "class": "connection_pool_exhaustion",
  "confidence": 0.9,
  "actions": [
    "Rollback to v3.1. Scale pool 50 \u2192 100 cushion. Add pool monitor alert > 80%."
  ],
  "reasoning": "Graph traversal picked payment-svc as most likely root cause. Retrieval matched 3 past incidents.",
  "similar_incidents": [
    "INC-2025-11-08",
    "INC-2026-03-20",
    "INC-2025-08-02"
  ],
  "method": "graph+retrieval"
}


## Bonus 1 — Decision Tree vs kNN Classifier
Training a `DecisionTreeClassifier` on the 30 historical incidents and comparing with the kNN heuristic.

In [13]:
# ====== Cell 5: Bonus 1 - Decision Tree vs kNN ======
import warnings
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')

all_services = sorted(list(set(svc for inc in incidents for svc in inc['services_involved'])))

def extract_features(inc):
    feats = [1 if svc in inc['services_involved'] else 0 for svc in all_services]
    sev_map = {'low': 1, 'medium': 2, 'high': 3, 'critical': 4}
    feats.append(sev_map.get(inc['severity'], 0))
    feats.append(inc.get('mttd_min', 0)) # Using mttd_min as a proxy for time_burst_pattern
    return feats

X = [extract_features(inc) for inc in incidents]
y = [inc['root_cause_class'] for inc in incidents]

X_train, X_test, y_train, y_test, inc_train, inc_test = train_test_split(
    X, y, incidents, test_size=0.3, random_state=42
)

# 1. Decision Tree
clf = DecisionTreeClassifier(random_state=42, max_depth=5)
clf.fit(X_train, y_train)
dt_preds = clf.predict(X_test)
dt_acc = accuracy_score(y_test, dt_preds)

# 2. kNN Heuristic (using our existing function)
knn_preds = []
for target_inc in inc_test:
    # Use retrieval from training incidents
    sim = retrieve_similar_incidents({"services": target_inc["services_involved"], "max_severity": target_inc["severity"]}, inc_train, top_k=1)
    if sim:
        knn_preds.append(sim[0][1]['root_cause_class'])
    else:
        knn_preds.append("other")
        
knn_acc = accuracy_score(y_test, knn_preds)

print("--- Bonus 1 Results (30% Test Set) ---")
print(f"Decision Tree Accuracy: {dt_acc:.2%}")
print(f"kNN Heuristic Accuracy: {knn_acc:.2%}")
print("\nInsight: Accuracy is extremely low for both because the dataset is tiny (30 incidents) and most root_cause_class labels are unique. DT fails completely on unseen classes, whereas kNN might occasionally find a heuristic overlap.")


--- Bonus 1 Results (30% Test Set) ---
Decision Tree Accuracy: 0.00%
kNN Heuristic Accuracy: 11.11%

Insight: Accuracy is extremely low for both because the dataset is tiny (30 incidents) and most root_cause_class labels are unique. DT fails completely on unseen classes, whereas kNN might occasionally find a heuristic overlap.
